In [0]:
%sql
DROP TABLE IF EXISTS demand;

CREATE TABLE if NOT Exists demand (
  product VARCHAR(10),
  day INT,
  qty FLOAT
);


INSERT INTO demand
  (product, day, qty)
VALUES
  ("A", 1, 10),
  ("A", 2, 6),
  ("A", 3, 21),
  ("A", 4, 9),
  ("A", 5, 19),
  ("B", 1, 12),
  ("B", 2, 18),
  ("B", 3, 3),
  ("B", 4, 6),
  ("B", 5, 23);

num_affected_rows,num_inserted_rows
10,10


In [0]:
%sql 

select * from demand;

product,day,qty
A,1,10.0
A,2,6.0
A,3,21.0
A,4,9.0
A,5,19.0
B,1,12.0
B,2,18.0
B,3,3.0
B,4,6.0
B,5,23.0


In [0]:
%sql
-- Q1. Find the running cumulative total demand by product.

select
*,
sum(qty) over (partition by product order by day ) as cumulative_sum
from demand;

product,day,qty,cumulative_sum
A,1,10.0,10.0
A,2,6.0,16.0
A,3,21.0,37.0
A,4,9.0,46.0
A,5,19.0,65.0
B,1,12.0,12.0
B,2,18.0,30.0
B,3,3.0,33.0
B,4,6.0,39.0
B,5,23.0,62.0


In [0]:
%sql
-- Q2. When are the top 2 worst performing days for each product?

select * from (select 
*,
row_number() over(partition by product order by qty) as rnk
from demand) where rnk <=2;

product,day,qty,rnk
A,2,6.0,1
A,4,9.0,2
B,3,3.0,1
B,4,6.0,2


In [0]:
%sql
-- Q3. Find the percentage increase in qty compared to the previous day.

select
s.product,
s.day,
s.qty,
s.qty_lag,
round(((s.qty-s.qty_lag)/s.qty_lag *100),2) as perc_increase from 
(select
*,
lag(qty) over (partition by product order by day) as qty_lag
from demand)s where s.qty_lag is not null;

product,day,qty,qty_lag,perc_increase
A,2,6.0,10.0,-40.0
A,3,21.0,6.0,250.0
A,4,9.0,21.0,-57.14
A,5,19.0,9.0,111.11
B,2,18.0,12.0,50.0
B,3,3.0,18.0,-83.33
B,4,6.0,3.0,100.0
B,5,23.0,6.0,283.33


In [0]:
%sql
-- Q4. Show the minimum and maximum ‘qty’ sold for each product as separate columns.
select * from demand d
join 
(select
product,
min(qty) as min_qty,
max(qty) as max_qty
from demand group by product)a on d.product = a.product ;

select *,
        min(qty) over(partition by product) as min_qty, 
        max(qty) over(partition by product) as max_qty 
from
demand;

product,day,qty,min_qty,max_qty
A,1,10.0,6.0,21.0
A,2,6.0,6.0,21.0
A,3,21.0,6.0,21.0
A,4,9.0,6.0,21.0
A,5,19.0,6.0,21.0
B,1,12.0,3.0,23.0
B,2,18.0,3.0,23.0
B,3,3.0,3.0,23.0
B,4,6.0,3.0,23.0
B,5,23.0,3.0,23.0


In [0]:
%sql
-- Q5. Calculate the difference between the second largest and the second smallest sales qty

select * from (select 
*,
row_number() over(partition by product order by qty) as rnk
from demand) where rnk = 2

union

select * from (select 
*,
row_number() over(partition by product order by qty desc) as rnk
from demand) where rnk = 2;


select product, 
 day, 
 qty
from
  (select *,
  row_number() over (partition by product order by qty) as rownum, 
  count(*) over (partition by product) as total_recs
  from
  demand) a
where (rownum = 2) or rownum = (total_recs - 1);

product,day,qty
A,4,9.0
A,5,19.0
B,4,6.0
B,2,18.0


In [0]:
%sql
-- Q8. On each day, which product had the highest sales?

with tbl as (
  select day, 
          product,
          qty,
          max(qty) over (partition by day) as maxqty
  from demand)
select day,
	product 
from tbl
where qty = maxqty order by day;

day,product
1,B
2,B
3,A
4,A
5,B
